# Cell-type-specific expression perturbation

Paper: Fig. 5 cell-type-specific strategy and the per-feature effects underlying Supplementary Fig. S6. Inputs: the original full-cohort joblib feature DataFrame and the trained final PeriClock artifact (trusted downloads only). Environment: Python, NumPy, pandas, SciPy/HiGHS, scikit-learn, joblib and Matplotlib. Outputs: per-age-group perturbation, full-results, individual-results, target-sample and summary CSVs, plus original lollipop plots below `OUTPUT_DIR`. The source analysis operates on saved training-space features; it is not a new normalization or predictor. It does not itself assemble all Fig. 5 panels or perform bootstrap comparisons.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Shared `PeriClock-1000s_PeriClock_features_matrix.pkl`, final PeriClock model and licensed Arial fonts; [training matrices](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices) and [models](../RESOURCE_DOWNLOAD.md#1-periclock-pretrained-models).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
FEATURES_1000 = Path(os.environ.get("PERICLOCK_FEATURES_1000", str(RESOURCE_DIR / "datasets/PeriClock-1000s_PeriClock_features_matrix.pkl"))).expanduser().resolve()
MODEL_PATH = Path(os.environ.get("PERICLOCK_PERTURBATION_MODEL", str(MODEL_DIR / "PeriClock.pkl"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/05_celltype_specific_perturbation"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [FEATURES_1000, MODEL_PATH]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Saved-model preprocessing, constrained optimization and original plotting

Source: cells 1, 3, 5, 7, 9 and 11, in their original order. Cell 7 runs the 50–60 group; cell 11 runs the other three groups. The 50–60 output remains in the top-level result directory, as in the source, while the other groups have subdirectories. No prior notebook state is required. The original HiGHS linear program, ten-year target, fixed proportions, SD weights, lower expression bounds and feature-selection thresholds remain unchanged. Expression percentages use inverse scaling and expm1 as in the source; they are not percentages of log1p expression. Inspect solver success before interpreting outputs. No optimization has been run during this preparation.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from scipy.optimize import linprog

# ============================================================
# 0. 路径
# ============================================================
feature_file = str(FEATURES_1000)
pipeline_file = str(MODEL_PATH)
output_dir = str(OUTPUT_DIR / "celltype_specific_perb_feature")
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# 1. Arial
# ============================================================
font_dir = str(FONT_DIR)
if os.path.isdir(font_dir):
    for f in os.listdir(font_dir):
        if f.lower().endswith((".ttf", ".otf")):
            fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 13, "font.weight": "bold",
    "axes.labelsize": 14, "axes.labelweight": "bold", "axes.titlesize": 15, "axes.titleweight": "bold",
    "xtick.labelsize": 11, "ytick.labelsize": 11, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.unicode_minus": False
})

# ============================================================
# 2. 读取
# ============================================================
df_all_features = joblib.load(feature_file)
pipeline = joblib.load(pipeline_file)

vt = pipeline["vt"]
skb = pipeline["skb"]
scaler_prop = pipeline["scaler_prop"]
scaler_expr = pipeline["scaler_expr"]
en_model = pipeline["model"]

prop_cols = [str(x) for x in pipeline["prop_cols"]]
expr_cols = [str(x) for x in pipeline["expr_cols"]]
selected_features = [str(x) for x in pipeline["selected_features"]]
print(len(selected_features), "selected features for ElasticNet model.")

# ============================================================
# 3. 年龄和特征
# ============================================================
required_cols = prop_cols + expr_cols + ["Age"]
missing_cols = [x for x in required_cols if x not in df_all_features.columns]
if missing_cols:
    raise ValueError("缺少模型所需列：\n" + "\n".join(missing_cols[:30]))

y_series = pd.to_numeric(df_all_features["Age"], errors="coerce")
valid_mask = y_series.notna()
df_model = df_all_features.loc[valid_mask].copy()
y_series = y_series.loc[valid_mask].astype(float)
y = y_series.to_numpy()

X_prop_raw = df_model[prop_cols].apply(pd.to_numeric, errors="coerce")
X_expr_raw = df_model[expr_cols].apply(pd.to_numeric, errors="coerce")

if X_prop_raw.isna().any().any():
    raise ValueError("比例特征存在NA或非数值。")
if X_expr_raw.isna().any().any():
    raise ValueError("表达特征存在NA或非数值。")

# ============================================================
# 4. 按训练pipeline处理expression
# ============================================================
X_expr_vt = vt.transform(X_expr_raw)
vt_expr_cols = np.asarray(expr_cols, dtype=str)[vt.get_support()]

X_expr_selected = skb.transform(X_expr_vt)
selected_expr_cols = [str(x) for x in vt_expr_cols[skb.get_support()]]

if X_expr_selected.shape[1] != len(scaler_expr.mean_):
    raise ValueError("表达特征数量与scaler_expr不一致。")
if X_prop_raw.shape[1] != len(scaler_prop.mean_):
    raise ValueError("比例特征数量与scaler_prop不一致。")

# ============================================================
# 5. StandardScaler
# ============================================================
X_prop_scaled = scaler_prop.transform(X_prop_raw)
X_expr_scaled = scaler_expr.transform(X_expr_selected)

all_cols = prop_cols + selected_expr_cols
X_final_df = pd.DataFrame(
    np.hstack([X_prop_scaled, X_expr_scaled]),
    columns=all_cols,
    index=df_model.index
)

# ============================================================
# 6. ElasticNet最终输入
# ============================================================
missing_selected = [x for x in selected_features if x not in X_final_df.columns]
if missing_selected:
    raise ValueError("部分selected_features不存在：\n" + "\n".join(missing_selected[:30]))

X = X_final_df.loc[:, selected_features].copy()
X.columns = X.columns.astype(str)

if hasattr(en_model, "feature_names_in_"):
    model_features = [str(x) for x in en_model.feature_names_in_]
    missing_model_features = [x for x in model_features if x not in X.columns]
    if missing_model_features:
        raise ValueError("X中缺少模型训练时特征：\n" + "\n".join(missing_model_features[:30]))
    X = X.loc[:, model_features].copy()

if X.shape[1] != len(en_model.coef_):
    raise ValueError(f"X特征数与ElasticNet系数数量不一致：{X.shape[1]} vs {len(en_model.coef_)}")

pred_age_all = en_model.predict(X)

# ============================================================
# 7. 只允许expression feature参与扰动
# ============================================================
selected_expr_set = set(selected_expr_cols)
model_expression_features = [f for f in X.columns if f in selected_expr_set]

print("Model input shape:", X.shape)
print("Expression features for perturbation:", len(model_expression_features))
print("Prediction range:", pred_age_all.min(), pred_age_all.max())

In [ ]:
# ============================================================
# 8. expression feature标准化空间下界
# ============================================================
def build_expression_lower_bounds(feature_names, scaler_expr, selected_expr_cols, min_expr=0.0):
    expr_mean = pd.Series(scaler_expr.mean_, index=selected_expr_cols, dtype=float)
    expr_scale = pd.Series(scaler_expr.scale_, index=selected_expr_cols, dtype=float)

    missing = [f for f in feature_names if f not in expr_mean.index]
    if missing:
        raise ValueError("以下feature没有scaler参数：\n" + "\n".join(missing[:20]))

    return (min_expr - expr_mean.loc[feature_names]) / expr_scale.loc[feature_names]


# ============================================================
# 9. Celltype-specific minimal perturbation
# ============================================================
def solve_celltype_specific_rejuvenation(
    X_target, en_model, scaler_expr, selected_expr_cols, candidate_features,
    reduce_years=10, max_change=1.0, min_expr=0.0, weight_by_sd=True,
    perturbation_tol=1e-6, age_effect_tol=0.01
):
    X_target = X_target.copy().astype(float)
    X_target.columns = X_target.columns.astype(str)

    candidate_features = [str(f) for f in candidate_features if str(f) in X_target.columns]
    beta_all = pd.Series(en_model.coef_, index=X_target.columns, dtype=float)
    beta = beta_all.loc[candidate_features]

    nonzero = beta.abs() > 1e-12
    candidate_features = beta.index[nonzero].tolist()
    beta = beta.loc[candidate_features]

    if len(candidate_features) == 0:
        raise ValueError("没有ElasticNet系数非零的expression feature。")

    X0 = X_target[candidate_features].copy()
    age_before = np.asarray(en_model.predict(X_target), dtype=float)
    n = len(candidate_features)

    # expression不能低于log1p=0
    lower_scaled = build_expression_lower_bounds(candidate_features, scaler_expr, selected_expr_cols, min_expr=min_expr)

    # feature perturbation weight
    if weight_by_sd:
        sd = X0.std().replace(0, np.nan)
        weights = (1.0 / sd).replace([np.inf, -np.inf], np.nan)
        if weights.notna().sum() == 0:
            weights = pd.Series(1.0, index=candidate_features)
        else:
            weights = weights.fillna(weights.median())
            weights = weights.clip(lower=weights.quantile(0.05), upper=weights.quantile(0.95))
    else:
        weights = pd.Series(1.0, index=candidate_features)

    # objective: min sum(weight * |delta|)
    c = np.concatenate([weights.values, weights.values])

    # beta * delta <= -reduce_years
    A_ub = np.concatenate([beta.values, -beta.values]).reshape(1, -1)
    b_ub = np.array([-reduce_years])

    # d_pos
    bounds = [(0.0, max_change)] * n

    # d_neg
    for f in candidate_features:
        max_down = (X0[f] - lower_scaled.loc[f]).min()
        max_down = max(0.0, min(max_change, max_down))
        bounds.append((0.0, max_down))

    # LP
    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

    if not res.success:
        raise RuntimeError("Optimization failed:\n" + res.message)

    d_pos = res.x[:n]
    d_neg = res.x[n:]
    delta = d_pos - d_neg

    # ========================================================
    # Full solution
    # ========================================================
    X_new_full = X_target.copy()
    X_new_full.loc[:, candidate_features] = X0.values + delta[np.newaxis, :]
    age_after_full = np.asarray(en_model.predict(X_new_full), dtype=float)

    age_effect = beta.values * delta

    full_result_df = pd.DataFrame({
        "feature": candidate_features,
        "beta": beta.values,
        "perturbation_scaled": delta,
        "abs_perturbation_scaled": np.abs(delta),
        "age_effect": age_effect,
        "age_reduction_contribution": -age_effect,
        "mean_x_scaled_original": X0.mean().values,
        "mean_x_scaled_new": X_new_full[candidate_features].mean().values,
        "lower_bound_scaled": lower_scaled.values,
        "weight": weights.values
    })

    full_result_df["direction"] = np.where(
        full_result_df["perturbation_scaled"] > 0, "up",
        np.where(full_result_df["perturbation_scaled"] < 0, "down", "none")
    )

    full_result_df = full_result_df.sort_values(
        "age_reduction_contribution", ascending=False
    ).reset_index(drop=True)

    # ========================================================
    # Core solution
    # ========================================================
    result_df = full_result_df.loc[
        (full_result_df["abs_perturbation_scaled"] > perturbation_tol) &
        (full_result_df["age_reduction_contribution"].abs() > age_effect_tol)
    ].copy()

    X_new_core = X_target.copy()

    if len(result_df) > 0:
        core_features = result_df["feature"].tolist()
        delta_series = pd.Series(delta, index=candidate_features)
        X_new_core.loc[:, core_features] = (
            X_target.loc[:, core_features].values +
            delta_series.loc[core_features].values[np.newaxis, :]
        )

    age_after_core = np.asarray(en_model.predict(X_new_core), dtype=float)

    reduction_full = age_before - age_after_full
    reduction_core = age_before - age_after_core

    individual_df = pd.DataFrame({
        "sample_id": X_target.index,
        "pred_age_before": age_before,
        "pred_age_after_full": age_after_full,
        "pred_age_after_core": age_after_core,
        "age_reduction_full": reduction_full,
        "age_reduction_core": reduction_core
    })

    summary = {
        "success": True,
        "n_samples": X_target.shape[0],
        "target_reduce_years": reduce_years,
        "n_candidate_expression_features": len(candidate_features),
        "n_features_full": int(np.sum(np.abs(delta) > perturbation_tol)),
        "n_features_core": int(result_df.shape[0]),
        "mean_age_before": float(age_before.mean()),
        "mean_age_after_full": float(age_after_full.mean()),
        "mean_age_after_core": float(age_after_core.mean()),
        "mean_age_reduction_full": float(reduction_full.mean()),
        "mean_age_reduction_core": float(reduction_core.mean()),
        "median_age_reduction_core": float(np.median(reduction_core)),
        "min_age_reduction_core": float(reduction_core.min()),
        "max_age_reduction_core": float(reduction_core.max()),
        "lost_reduction_after_filtering": float(reduction_full.mean() - reduction_core.mean()),
        "weighted_total_perturbation": float(
            (result_df["abs_perturbation_scaled"] * result_df["weight"]).sum()
        ),
        "optimization_objective": float(res.fun)
    }

    return {
        "perturbation_df": result_df,
        "full_result_df": full_result_df,
        "individual_result_df": individual_df,
        "summary": summary,
        "X_new": X_new_core,
        "X_new_full": X_new_full,
        "optimization_result": res
    }

In [ ]:
# ============================================================
# 10. 标准化扰动 -> linear-expression fold change
# ============================================================
def add_linear_expression_change(result_df, X_target, scaler_expr, selected_expr_cols, eps=1e-8):
    result_df = result_df.copy()

    expr_mean = pd.Series(scaler_expr.mean_, index=selected_expr_cols, dtype=float)
    expr_scale = pd.Series(scaler_expr.scale_, index=selected_expr_cols, dtype=float)

    records = {}

    for f in result_df["feature"]:
        if f not in expr_mean.index:
            raise ValueError(f"{f}不是scaler_expr对应的expression feature。")

        z_old = X_target[f].astype(float).to_numpy()
        delta_scaled = float(
            result_df.loc[result_df["feature"] == f, "perturbation_scaled"].iloc[0]
        )

        # standardized -> log1p
        log1p_old = z_old * expr_scale.loc[f] + expr_mean.loc[f]
        delta_log1p = delta_scaled * expr_scale.loc[f]
        log1p_new = log1p_old + delta_log1p

        if np.min(log1p_old) < -1e-6:
            raise ValueError(
                f"{f}反标准化后出现明显负值：min={np.min(log1p_old):.6f}，请确认scaler前是否为log1p表达。"
            )

        if np.min(log1p_new) < -1e-6:
            raise ValueError(
                f"{f}扰动后log1p表达低于0：min={np.min(log1p_new):.6f}"
            )

        log1p_old = np.clip(log1p_old, 0, None)
        log1p_new = np.clip(log1p_new, 0, None)

        # log1p -> linear
        linear_old = np.expm1(log1p_old)
        linear_new = np.expm1(log1p_new)

        mean_old = float(np.mean(linear_old))
        mean_new = float(np.mean(linear_new))

        if mean_old > eps:
            fold_change_linear = mean_new / mean_old
            percent_change_linear = (fold_change_linear - 1.0) * 100
        else:
            fold_change_linear = np.nan
            percent_change_linear = np.nan

        # donor-level FC，用于诊断
        valid = linear_old > eps

        if np.sum(valid) > 0:
            sample_fc = linear_new[valid] / linear_old[valid]
            sample_percent = (sample_fc - 1.0) * 100
            median_sample_fc = float(np.median(sample_fc))
            median_sample_percent = float(np.median(sample_percent))
            q25_sample_percent = float(np.quantile(sample_percent, 0.25))
            q75_sample_percent = float(np.quantile(sample_percent, 0.75))
        else:
            median_sample_fc = np.nan
            median_sample_percent = np.nan
            q25_sample_percent = np.nan
            q75_sample_percent = np.nan

        records[f] = {
            "mean_log1p_before": float(np.mean(log1p_old)),
            "mean_log1p_after": float(np.mean(log1p_new)),
            "delta_log1p": float(delta_log1p),
            "mean_linear_before": mean_old,
            "mean_linear_after": mean_new,
            "fold_change_linear": fold_change_linear,
            "percent_change_linear": percent_change_linear,
            "median_sample_fold_change": median_sample_fc,
            "median_sample_percent_change": median_sample_percent,
            "q25_sample_percent_change": q25_sample_percent,
            "q75_sample_percent_change": q75_sample_percent
        }

    for col in [
        "mean_log1p_before", "mean_log1p_after", "delta_log1p",
        "mean_linear_before", "mean_linear_after", "fold_change_linear",
        "percent_change_linear", "median_sample_fold_change",
        "median_sample_percent_change", "q25_sample_percent_change",
        "q75_sample_percent_change"
    ]:
        result_df[col] = result_df["feature"].map(lambda f: records[f][col])

    result_df["percent_change_label"] = result_df["percent_change_linear"].apply(
        lambda x: np.nan if pd.isna(x) else f"{x:+.1f}%"
    )

    return result_df

In [ ]:
# ============================================================
# 11. 总pipeline
# ============================================================
def run_celltype_specific_pipeline(
    X, y, en_model, scaler_expr, selected_expr_cols, candidate_features,
    age_range=(50, 60), reduce_years=10, max_change=1.0, min_expr=0.0,
    weight_by_sd=True, perturbation_tol=1e-6, age_effect_tol=0.01,
    save_prefix=None
):
    pred_age = pd.Series(en_model.predict(X), index=X.index, name="pred_age")
    y_series = pd.Series(y, index=X.index, name="true_age")

    low, high = age_range
    target_mask = (pred_age >= low) & (pred_age < high)
    X_target = X.loc[target_mask].copy()

    if X_target.shape[0] == 0:
        raise ValueError(f"预测年龄{low}–{high}岁范围内没有样本。")

    print(f"Target predicted age: {low}-{high}")
    print("N samples:", X_target.shape[0])
    print("Mean predicted age:", pred_age.loc[X_target.index].mean())

    result = solve_celltype_specific_rejuvenation(
        X_target=X_target,
        en_model=en_model,
        scaler_expr=scaler_expr,
        selected_expr_cols=selected_expr_cols,
        candidate_features=candidate_features,
        reduce_years=reduce_years,
        max_change=max_change,
        min_expr=min_expr,
        weight_by_sd=weight_by_sd,
        perturbation_tol=perturbation_tol,
        age_effect_tol=age_effect_tol
    )

    result["perturbation_df"] = add_linear_expression_change(
        result["perturbation_df"], X_target, scaler_expr, selected_expr_cols
    )

    result["full_result_df"] = add_linear_expression_change(
        result["full_result_df"], X_target, scaler_expr, selected_expr_cols
    )

    individual_df = result["individual_result_df"].copy()
    individual_df.insert(
        1,
        "true_age",
        y_series.loc[individual_df["sample_id"]].values
    )

    target_info_df = pd.DataFrame({
        "sample_id": X_target.index,
        "true_age": y_series.loc[X_target.index].values,
        "pred_age_before": pred_age.loc[X_target.index].values
    })

    result["individual_result_df"] = individual_df
    result["target_info_df"] = target_info_df
    result["X_target"] = X_target

    if save_prefix is not None:
        result["perturbation_df"].to_csv(f"{save_prefix}_perturbation.csv", index=False)
        result["full_result_df"].to_csv(f"{save_prefix}_full_results.csv", index=False)
        result["individual_result_df"].to_csv(f"{save_prefix}_individual_results.csv", index=False)
        result["target_info_df"].to_csv(f"{save_prefix}_target_samples.csv", index=False)
        pd.DataFrame([result["summary"]]).to_csv(f"{save_prefix}_summary.csv", index=False)

    return result


# ============================================================
# 12. 运行50–60岁 -> 下降10岁
# ============================================================
save_prefix = os.path.join(
    output_dir,
    "50-60_10yr_celltype_specific"
)

result_50_60 = run_celltype_specific_pipeline(
    X=X,
    y=y,
    en_model=en_model,
    scaler_expr=scaler_expr,
    selected_expr_cols=selected_expr_cols,
    candidate_features=model_expression_features,
    age_range=(50, 60),
    reduce_years=10,
    max_change=1.0,
    min_expr=0.0,
    weight_by_sd=True,
    perturbation_tol=1e-6,
    age_effect_tol=0.01,
    save_prefix=save_prefix
)

print("\n================ SUMMARY ================")
for k, v in result_50_60["summary"].items():
    print(f"{k}: {v}")

print("\nTop perturbations:")
print(
    result_50_60["perturbation_df"][
        [
            "feature",
            "beta",
            "perturbation_scaled",
            "fold_change_linear",
            "percent_change_linear",
            "age_reduction_contribution"
        ]
    ].head(30)
)

In [ ]:
from matplotlib.lines import Line2D
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Celltype-specific perturbation lollipop
# 数值统一标在圆点正上方
# ============================================================
def plot_celltype_specific_perturbation(perturbation_df, age_range=(50, 60), top_n=50, output_prefix=None):

    plot_df = perturbation_df.copy()
    required_cols = ["feature", "percent_change_linear", "age_reduction_contribution"]
    missing = [x for x in required_cols if x not in plot_df.columns]
    if missing:
        raise ValueError("缺少作图列：" + ", ".join(missing))

    # 只保留真正降低预测年龄的feature
    plot_df = plot_df.loc[plot_df["age_reduction_contribution"] > 0].copy()

    # Top N
    plot_df = (
        plot_df.sort_values("age_reduction_contribution", ascending=False)
        .head(top_n)
        .sort_values("age_reduction_contribution", ascending=True)
        .reset_index(drop=True)
    )

    if len(plot_df) == 0:
        raise ValueError("没有可绘制的年轻化扰动feature。")

    plot_df["direction_plot"] = np.where(plot_df["percent_change_linear"] > 0, "Up", "Down")

    # ========================================================
    # 点大小
    # ========================================================
    contribution = plot_df["age_reduction_contribution"].to_numpy(dtype=float)
    min_size, max_size = 80, 420

    if contribution.max() > contribution.min():
        point_sizes = min_size + (contribution - contribution.min()) / (contribution.max() - contribution.min()) * (max_size - min_size)
    else:
        point_sizes = np.full(len(plot_df), (min_size + max_size) / 2)

    # 低饱和配色
    up_color = "#B65D52"
    down_color = "#526F8F"
    point_colors = np.where(plot_df["direction_plot"].eq("Up"), up_color, down_color)

    # ========================================================
    # Figure
    # 稍微增加每个feature之间的纵向间距
    # ========================================================
    row_spacing = 1.18
    y_pos = np.arange(len(plot_df)) * row_spacing

    fig_height = max(5.8, 0.48 * len(plot_df) + 1.8)
    fig, ax = plt.subplots(figsize=(8.2, fig_height))

    # ========================================================
    # Lollipop横线
    # ========================================================
    for y, (_, row) in zip(y_pos, plot_df.iterrows()):
        value = row["percent_change_linear"]
        ax.hlines(
            y=y,
            xmin=min(0, value),
            xmax=max(0, value),
            color=up_color if value > 0 else down_color,
            linewidth=2.3,
            alpha=0.72,
            zorder=1
        )

    # ========================================================
    # 圆点
    # ========================================================
    ax.scatter(
        plot_df["percent_change_linear"],
        y_pos,
        s=point_sizes,
        c=point_colors,
        edgecolor="white",
        linewidth=1.2,
        alpha=0.95,
        zorder=3
    )

    # 0参考线
    ax.axvline(0, color="#333333", linewidth=1.2, linestyle="--", zorder=0)

    # ========================================================
    # Y轴feature
    # ========================================================
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["feature"], fontsize=11, fontweight="bold")

    # ========================================================
    # 数值标注
    # 改为：圆点正上方
    # ========================================================
    for y, (_, row) in zip(y_pos, plot_df.iterrows()):
        x = row["percent_change_linear"]
        contribution_value = row["age_reduction_contribution"]

        ax.annotate(
            f"{x:+.1f}%  |  {contribution_value:.2f} yr",
            xy=(x, y),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9.2,
            fontweight="bold",
            color="#222222",
            zorder=5
        )

    # ========================================================
    # X范围
    # 因为文字不再放左右侧，所以不需要很大的左右留白
    # ========================================================
    x_min = min(plot_df["percent_change_linear"].min(), 0)
    x_max = max(plot_df["percent_change_linear"].max(), 0)
    x_range = max(x_max - x_min, 1)

    ax.set_xlim(x_min - 0.15 * x_range, x_max + 0.15 * x_range)
    ax.set_ylim(-0.65 * row_spacing, y_pos[-1] + 0.75 * row_spacing)

    # ========================================================
    # 坐标轴
    # ========================================================
    ax.set_xlabel("Required gene expression change (%)", fontsize=14, fontweight="bold")
    ax.set_ylabel("")

    low, high = age_range

    ax.text(
        0.00, 1.08, f"{low}–{high} years",
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=14, fontweight="bold"
    )

    ax.text(
        0.01, 1.015, "Down-regulation",
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=10.5, color=down_color,
        fontweight="bold"
    )

    ax.text(
        0.99, 1.015, "Up-regulation",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=10.5, color=up_color,
        fontweight="bold"
    )

    # ========================================================
    # Direction legend
    # ========================================================
    direction_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=down_color, markeredgecolor="white", markersize=9, label="Down"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor=up_color, markeredgecolor="white", markersize=9, label="Up")
    ]

    direction_legend = ax.legend(
        handles=direction_handles,
        title="Direction",
        loc="lower right",
        bbox_to_anchor=(1.0, 0.0),
        frameon=False,
        fontsize=10,
        title_fontsize=10.5
    )

    ax.add_artist(direction_legend)

    # ========================================================
    # 点大小图例
    # ========================================================
    size_values = np.unique(np.round(np.quantile(contribution, [0.25, 0.50, 0.90]), 2))
    size_handles = []

    for value in size_values:
        if contribution.max() > contribution.min():
            marker_size = min_size + (value - contribution.min()) / (contribution.max() - contribution.min()) * (max_size - min_size)
        else:
            marker_size = (min_size + max_size) / 2

        size_handles.append(
            plt.scatter([], [], s=marker_size, facecolor="#777777", edgecolor="white", linewidth=1, label=f"{value:.2f}")
        )

    ax.legend(
        handles=size_handles,
        title="Age reduction\ncontribution (years)",
        loc="lower right",
        bbox_to_anchor=(1.0, 0.14),
        frameon=False,
        fontsize=9.5,
        title_fontsize=10
    )

    # ========================================================
    # Style
    # ========================================================
    ax.grid(axis="x", linestyle="--", linewidth=0.7, alpha=0.25)
    ax.grid(axis="y", visible=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=11)

    plt.tight_layout()

    # ========================================================
    # Save
    # ========================================================
    if output_prefix is not None:
        plt.savefig(f"{output_prefix}.pdf", bbox_inches="tight")
        plt.savefig(f"{output_prefix}.png", dpi=600, bbox_inches="tight")

    plt.show()


# ============================================================
# 作图
# ============================================================
plot_celltype_specific_perturbation(
    perturbation_df=result_50_60["perturbation_df"],
    age_range=(50, 60),
    top_n=50,
    output_prefix=os.path.join(output_dir, "50-60_10yr_celltype_specific_lollipop")
)

In [ ]:
import os

# ============================================================
# 1. 总输出目录
# ============================================================
base_output_dir = str(OUTPUT_DIR / "celltype_specific_perb_feature")

# 需要计算的年龄段
age_groups = [(40, 50), (60, 70), (70, 90)]

# 保存每个年龄段的结果对象，方便后面直接调用
all_results = {}

# ============================================================
# 2. 批量计算扰动 + 作图
# ============================================================
for low, high in age_groups:
    print("\n" + "=" * 70)
    print(f"Running predicted age group: {low}-{high} years")
    print("=" * 70)

    # 每个年龄段单独文件夹
    age_output_dir = os.path.join(base_output_dir, f"{low}-{high}")
    os.makedirs(age_output_dir, exist_ok=True)

    # 文件名前缀
    save_prefix = os.path.join(age_output_dir, f"{low}-{high}_10yr_celltype_specific")

    # ========================================================
    # 扰动优化
    # ========================================================
    result = run_celltype_specific_pipeline(
        X=X,
        y=y,
        en_model=en_model,
        scaler_expr=scaler_expr,
        selected_expr_cols=selected_expr_cols,
        candidate_features=model_expression_features,
        age_range=(low, high),
        reduce_years=10,
        max_change=1.0,
        min_expr=0.0,
        weight_by_sd=True,
        perturbation_tol=1e-6,
        age_effect_tol=0.01,
        save_prefix=save_prefix
    )

    all_results[f"{low}-{high}"] = result

    # ========================================================
    # 打印summary
    # ========================================================
    print(f"\nSummary: {low}-{high} years")
    for k, v in result["summary"].items():
        print(f"{k}: {v}")

    # ========================================================
    # 作图
    # ========================================================
    plot_celltype_specific_perturbation(
        perturbation_df=result["perturbation_df"],
        age_range=(low, high),
        top_n=50,
        output_prefix=os.path.join(age_output_dir, f"{low}-{high}_10yr_celltype_specific_lollipop")
    )

    print(f"\nFinished: {low}-{high}")
    print("Saved to:", age_output_dir)